# C7-cnn-transfer — Practice p19 — Solution

The helper assigns every flag explicitly in one pass, so its result does not depend on the flags a copied or newly grafted module had beforehand.

In [ ]:
# Cache pin (course convention, plan 009): pretrained weights live in the repo's
# gitignored reference/cache/ -- resolve it from the repo root BEFORE importing torch.
import os, pathlib
_env_root = os.environ.get("USAAIO_BOOK_ROOT")
if _env_root:
    _root = pathlib.Path(_env_root).resolve()
else:
    _start = pathlib.Path.cwd().resolve()
    _root = next(
        p for p in [_start, *_start.parents]
        if (p / "syllabus.md").is_file() and (p / "curriculum").is_dir()
    )
os.environ["TORCH_HOME"] = str(_root / "reference" / "cache" / "torch")

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# float32 register (course exception): pretrained resnet50 is a float32 artifact.
# No float64 default here; inputs are cast .to(torch.float32) at the model
# boundary; repeat float32 forwards are bit-identical.
SEED = 20260804

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval()
assert next(model.parameters()).dtype == torch.float32

import copy
torch.manual_seed(SEED)
x = torch.randn(2, 3, 224, 224).to(torch.float32)

def freeze_except(module, prefixes):
    for name, p in module.named_parameters():
        p.requires_grad = name.startswith(prefixes)

surg = copy.deepcopy(model)
surg.fc = nn.Linear(2048, 17)
freeze_except(surg, ("layer4", "fc"))
surg.eval()


In [ ]:
# layer4 + fresh head = 14,964,736 + 2049*17.
hand_trainable = 14_999_569
# Original total - old 1000-class head - layer4.
hand_frozen = 25_557_032 - 2_049_000 - 14_964_736


In [ ]:
n_trainable = sum(p.numel() for p in surg.parameters() if p.requires_grad)
n_frozen = sum(p.numel() for p in surg.parameters() if not p.requires_grad)
counts_match = n_trainable == hand_trainable and n_frozen == hand_frozen
trainable_tops = sorted({name.split(".", 1)[0] for name, p in surg.named_parameters() if p.requires_grad})
with torch.inference_mode():
    out_shape = tuple(surg(x).shape)


The deepest stage carries the most task-specific, large-receptive-field
features, so `layer4` is the natural body region to adjust when sufficient
resources exist beyond this construction exercise. For the smallest projects,
the fresh head alone is the safer and much smaller adjustable surface.


### Answer check

In [ ]:
assert freeze_except(surg, ("layer4", "fc")) is None
assert (hand_trainable, hand_frozen) == (14_999_569, 8_543_296)
assert counts_match
assert trainable_tops == ["fc", "layer4"]
assert out_shape == (2, 17)
